# Decision Tree — C4.5

C4.5 (Quinlan, 1993) is the successor of ID3. Main improvements:

| Problem in ID3 | C4.5 solution |
|---|---|
| Information gain favours many-valued attributes | **Gain ratio** = IG / SplitInfo |
| Only categorical features | **Continuous features** via binary threshold splits $x \le t$ |
| Overfitting | **Pessimistic (error-based) post-pruning** |

**Split information** penalises attributes that split the data into many small parts:
$$\text{SplitInfo}(S, A) = -\sum_v \frac{|S_v|}{|S|}\log_2\frac{|S_v|}{|S|}, \qquad \text{GainRatio}(S,A) = \frac{IG(S,A)}{\text{SplitInfo}(S,A)}$$

Quinlan's heuristic: only consider attributes whose gain is at least the average gain, then choose the one with the
highest gain ratio (avoids picking attributes with tiny SplitInfo).

**Continuous attributes**: sort the values, try thresholds at midpoints between consecutive distinct values,
and keep the threshold with the best gain.

**Pessimistic pruning**: for a node with $N$ samples and $E$ training errors, C4.5 estimates the true error rate as the
upper limit $U_{CF}(E, N)$ of a binomial confidence interval (default $CF = 25\%$): the $p$ with $P(X \le E \mid N, p) = CF$. A subtree is replaced by a leaf if the leaf's estimated
errors $N\cdot U_{CF}$ are not larger than the sum of the estimated errors of the subtree's leaves.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

from math import comb

In [ ]:
def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Building blocks

In [ ]:
def entropy(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))


def split_info(sizes):
    sizes = np.asarray(sizes, dtype=float)
    p = sizes[sizes > 0] / sizes.sum()
    return -np.sum(p * np.log2(p))


def binom_cdf(k, n, p):
    return sum(comb(n, i) * p**i * (1 - p)**(n - i) for i in range(k + 1))


def pessimistic_error(n, errors, cf=0.25):
    """Estimated number of errors = n * U_CF(errors, n), where U_CF is the upper limit of the
    binomial confidence interval: the error rate p with P(X <= errors | n, p) = CF."""
    if n == 0:
        return 0.0
    if errors == 0:
        return n * (1 - cf ** (1 / n))
    lo, hi = errors / n, 1.0
    for _ in range(50):                      # bisection: binom_cdf decreases as p grows
        mid = (lo + hi) / 2
        if binom_cdf(errors, n, mid) > cf:
            lo = mid
        else:
            hi = mid
    return n * (lo + hi) / 2

## C4.5 implementation

In [ ]:
class C45:
    """C4.5 decision tree supporting categorical and continuous features.

    feature_types: list with "categorical" or "continuous" for each column.
    Nodes are dicts. Leaves: {"leaf": True, ...}; categorical split: "threshold" is None and
    "children" maps value -> subtree; continuous split: children keys are "<=" and ">".
    """

    def __init__(self, feature_types, max_depth=None, min_leaf=2, prune=True, cf=0.25):
        self.feature_types = feature_types
        self.max_depth = max_depth
        self.min_leaf = min_leaf      # C4.5's "-m": at least two branches need >= min_leaf samples
        self.prune = prune
        self.cf = cf                  # confidence factor for pruning (smaller = more pruning)

    # ------------------------------------------------------------------ fitting
    def fit(self, X, y, feature_names=None):
        X, y = np.asarray(X, dtype=object), np.asarray(y)
        self.feature_names = feature_names or [f"x{i}" for i in range(X.shape[1])]
        self.tree = self._build(X, y, depth=0)
        if self.prune:
            self.tree = self._prune(self.tree)
        return self

    def _leaf(self, y):
        values, counts = np.unique(y, return_counts=True)
        return {"leaf": True, "prediction": values[np.argmax(counts)],
                "n": len(y), "errors": len(y) - counts.max(),
                "counts": dict(zip(values.tolist(), counts.tolist()))}

    def _best_split(self, X, y):
        n, base = len(y), entropy(y)
        candidates = []                        # (gain, gain_ratio, feature, threshold)
        for f in range(X.shape[1]):
            col = X[:, f]
            if self.feature_types[f] == "categorical":
                values = np.unique(col)
                subsets = [y[col == v] for v in values]
                if sum(len(s) >= self.min_leaf for s in subsets) < 2:
                    continue
                gain = base - sum(len(s) / n * entropy(s) for s in subsets)
                si = split_info([len(s) for s in subsets])
                candidates.append((gain, gain / si, f, None))
            else:
                col = col.astype(float)
                values = np.unique(col)
                best = None
                for t in (values[:-1] + values[1:]) / 2:
                    left, right = y[col <= t], y[col > t]
                    if min(len(left), len(right)) < self.min_leaf:
                        continue
                    gain = base - (len(left) * entropy(left) + len(right) * entropy(right)) / n
                    if best is None or gain > best[0]:
                        best = (gain, gain / split_info([len(left), len(right)]), f, t)
                if best is not None:
                    candidates.append(best)

        candidates = [c for c in candidates if c[0] > 1e-12]
        if not candidates:
            return None
        avg_gain = np.mean([c[0] for c in candidates])
        good = [c for c in candidates if c[0] >= avg_gain - 1e-12]
        return max(good, key=lambda c: c[1])

    def _build(self, X, y, depth):
        node = self._leaf(y)
        if (len(node["counts"]) == 1 or len(y) < 2 * self.min_leaf
                or (self.max_depth is not None and depth >= self.max_depth)):
            return node

        split = self._best_split(X, y)
        if split is None:
            return node
        gain, ratio, f, t = split
        node.update(leaf=False, feature=f, threshold=t, gain_ratio=ratio, children={})
        if t is None:
            for v in np.unique(X[:, f]):
                mask = X[:, f] == v
                node["children"][v] = self._build(X[mask], y[mask], depth + 1)
        else:
            mask = X[:, f].astype(float) <= t
            node["children"]["<="] = self._build(X[mask], y[mask], depth + 1)
            node["children"][">"] = self._build(X[~mask], y[~mask], depth + 1)
        return node

    # ------------------------------------------------------------------ pruning
    def _subtree_error(self, node):
        if node["leaf"]:
            return pessimistic_error(node["n"], node["errors"], self.cf)
        return sum(self._subtree_error(c) for c in node["children"].values())

    def _prune(self, node):
        if node["leaf"]:
            return node
        for k in node["children"]:
            node["children"][k] = self._prune(node["children"][k])
        as_leaf = pessimistic_error(node["n"], node["errors"], self.cf)
        if as_leaf <= self._subtree_error(node) + 1e-9:
            return {"leaf": True, "prediction": node["prediction"], "n": node["n"],
                    "errors": node["errors"], "counts": node["counts"]}
        return node

    # ------------------------------------------------------------------ inference
    def _predict_one(self, x):
        node = self.tree
        while not node["leaf"]:
            f, t = node["feature"], node["threshold"]
            if t is None:
                child = node["children"].get(x[f])
                if child is None:              # unseen category
                    return node["prediction"]
                node = child
            else:
                node = node["children"]["<=" if float(x[f]) <= t else ">"]
        return node["prediction"]

    def predict(self, X):
        return np.array([self._predict_one(x) for x in np.asarray(X, dtype=object)])

    def count_leaves(self, node=None):
        node = self.tree if node is None else node
        return 1 if node["leaf"] else sum(self.count_leaves(c) for c in node["children"].values())

    def print_tree(self, node=None, indent=""):
        node = self.tree if node is None else node
        if node["leaf"]:
            print(f"{indent}-> {node['prediction']}  {node['counts']}")
            return
        name = self.feature_names[node["feature"]]
        for key, child in node["children"].items():
            cond = f"{name} = {key}" if node["threshold"] is None else f"{name} {key} {node['threshold']:.2f}"
            print(f"{indent}[{cond}]")
            self.print_tree(child, indent + "    ")

## Example 1 — Play Tennis with numeric Temperature and Humidity
This is the version of the dataset from Quinlan's C4.5 book.

In [ ]:
feature_names = ["Outlook", "Temperature", "Humidity", "Windy"]
feature_types = ["categorical", "continuous", "continuous", "categorical"]
X = np.array([
    ["sunny", 85, 85, "false"], ["sunny", 80, 90, "true"], ["overcast", 83, 86, "false"],
    ["rainy", 70, 96, "false"], ["rainy", 68, 80, "false"], ["rainy", 65, 70, "true"],
    ["overcast", 64, 65, "true"], ["sunny", 72, 95, "false"], ["sunny", 69, 70, "false"],
    ["rainy", 75, 80, "false"], ["sunny", 75, 70, "true"], ["overcast", 72, 90, "true"],
    ["overcast", 81, 75, "false"], ["rainy", 71, 91, "true"],
], dtype=object)
y = np.array(["no", "no", "yes", "yes", "yes", "no", "yes", "no", "yes", "yes", "yes", "yes", "yes", "no"])

tree = C45(feature_types, prune=False).fit(X, y, feature_names)
tree.print_tree()
print("\nTraining accuracy:", np.mean(tree.predict(X) == y))

### Gain vs gain ratio: the "ID column" problem
Add a unique `Day` identifier. Information gain loves it (it splits the data perfectly), gain ratio does not.

In [ ]:
day_id = np.arange(len(y)).astype(str)
base = entropy(y)
ig_day = base - sum((day_id == d).mean() * entropy(y[day_id == d]) for d in day_id)
ig_out = base - sum((X[:, 0] == v).mean() * entropy(y[X[:, 0] == v]) for v in np.unique(X[:, 0]))
si_day = split_info(np.ones(len(y)))
si_out = split_info([np.sum(X[:, 0] == v) for v in np.unique(X[:, 0])])

print(f"Day     : gain = {ig_day:.3f}, split info = {si_day:.3f}, gain ratio = {ig_day / si_day:.3f}")
print(f"Outlook : gain = {ig_out:.3f}, split info = {si_out:.3f}, gain ratio = {ig_out / si_out:.3f}")

## Example 2 — continuous data (two moons) and the effect of pruning

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(Xm, ym, test_size=0.3, random_state=0)
types = ["continuous", "continuous"]

unpruned = C45(types, prune=False).fit(X_train, y_train)
pruned = C45(types, prune=True).fit(X_train, y_train)

for name, m in [("unpruned", unpruned), ("pruned", pruned)]:
    print(f"{name:9s}: leaves = {m.count_leaves():3d}, "
          f"train acc = {accuracy_score(y_train, m.predict(X_train)):.3f}, "
          f"test acc = {accuracy_score(y_test, m.predict(X_test)):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(unpruned, X_train, y_train, f"Unpruned C4.5 ({unpruned.count_leaves()} leaves)", axes[0], resolution=100)
plot_decision_boundary(pruned, X_train, y_train, f"Pruned C4.5 ({pruned.count_leaves()} leaves)", axes[1], resolution=100)
plt.tight_layout(); plt.show()

In [ ]:
depths = range(1, 13)
tr, te = [], []
for d in depths:
    m = C45(types, max_depth=d, prune=False).fit(X_train, y_train)
    tr.append(accuracy_score(y_train, m.predict(X_train)))
    te.append(accuracy_score(y_test, m.predict(X_test)))
plt.plot(depths, tr, "o-", label="train"); plt.plot(depths, te, "s-", label="test")
plt.xlabel("max depth"); plt.ylabel("accuracy"); plt.legend(); plt.title("C4.5: depth vs accuracy"); plt.show()

## Summary
* **Gain ratio** corrects ID3's bias toward many-valued attributes.
* **Thresholds** let C4.5 handle continuous features directly.
* **Pessimistic pruning** removes branches that do not reduce the estimated error, giving smaller trees that usually generalise better.
* (Full C4.5 also handles missing values by distributing samples fractionally across branches — omitted here for clarity.)